# Execution calibration: independent accounting review

This companion checks a private canonical order run without using the measurement functions. Synthetic and paper evidence never establishes actual slippage or causal impact. Set `EXECUTION_COST_CALIBRATION_INPUT` and `EXECUTION_COST_CALIBRATION_RUN`; save executed outputs outside Git. Dependencies: pandas, pyarrow, nbformat, nbclient, ipykernel.

In [ ]:
import os,json,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
source=Path(os.environ['EXECUTION_COST_CALIBRATION_INPUT'])
run=Path(os.environ['EXECUTION_COST_CALIBRATION_RUN'])
receipt=json.loads((run/'private-receipt.json').read_text())
public=json.loads((run/'aggregate.json').read_text())
frames={}
for name in ('orders','fills','quotes'):
    paths=[source/(name+suffix) for suffix in ('.csv','.parquet')]
    path=next(p for p in paths if p.exists())
    assert hashlib.sha256(path.read_bytes()).hexdigest()==receipt['input_sha256'][name]
    frames[name]=pd.read_parquet(path) if path.suffix=='.parquet' else pd.read_csv(path,dtype={k:str for k in ('symbol','parent_order_id','fill_id')})
assert hashlib.sha256((source/'manifest.json').read_bytes()).hexdigest()==receipt['input_sha256']['manifest']
for name,expected in receipt['output_sha256'].items():
    assert hashlib.sha256((run/name).read_bytes()).hexdigest()==expected
measured=pd.read_parquet(run/'private-order-measurements.parquet')
assert len(measured)==len(frames['orders'])==public['orders']
assert measured.parent_order_id.is_unique
if set(measured.data_kind)!= {'actual'}:
    assert public['actual_order_calibrated'] is False
assert public['causal_own_impact_identified'] is False
print({'orders':len(measured),'data_kinds':public['data_kinds'],'actual_calibrated':public['actual_order_calibrated']})

In [ ]:
o,f,q=frames['orders'].copy(),frames['fills'].copy(),frames['quotes'].copy()
for col in ('decision_time','arrival_time','end_time'):
    o[col]=pd.to_datetime(o[col],utc=True)
f['fill_time']=pd.to_datetime(f.fill_time,utc=True)
q['quote_time']=pd.to_datetime(q.quote_time,utc=True)
q['available_time']=pd.to_datetime(q.get('available_time',q.quote_time),utc=True)
checks=[]
for order in o.itertuples(index=False):
    fills=f.loc[f.parent_order_id.eq(order.parent_order_id)]
    quantity=float(fills.qty.sum())
    assert quantity<=order.target_qty
    assert fills.fill_time.between(order.arrival_time,order.end_time).all()
    mids={}
    for key in ('decision','arrival','end'):
        t=getattr(order,key+'_time')
        past=q.loc[q.symbol.eq(order.symbol)&q.quote_time.le(t)&q.available_time.le(t)].sort_values('quote_time')
        mids[key]=None if past.empty or (t-past.iloc[-1].quote_time).total_seconds()*1000>public['definitions']['max_quote_age_ms'] else float((past.iloc[-1].bid+past.iloc[-1].ask)/2)
    row=measured.loc[measured.parent_order_id.eq(order.parent_order_id)].iloc[0]
    assert row.calibration_eligible==all(mid is not None for mid in mids.values())
    assert np.isclose(row.filled_qty,quantity)
    assert np.isclose(row.unfilled_qty,order.target_qty-quantity)
    if row.calibration_eligible:
        sign=1 if order.side=='buy' else -1
        execution=sign*float((fills.qty*(fills.price-mids['decision'])).sum())
        opportunity=sign*(order.target_qty-quantity)*(mids['end']-mids['decision'])
        expected=10000*(execution+opportunity+order.fees_cny)/(order.target_qty*mids['decision'])
        assert np.isclose(row.shortfall_bp,expected,rtol=1e-9,atol=1e-8)
        assert np.isclose(row.decision_execution_cny,execution,atol=1e-8)
        assert np.isclose(row.opportunity_cny,opportunity,atol=1e-8)
        checks.append(expected)
    else:
        assert pd.isna(row.shortfall_bp)
assert len(checks)==public['eligible_orders']
print({'independently_reconciled_parents':len(checks)})

In [ ]:
validation=json.loads((run/'private-validation.json').read_text())
valid=measured.loc[measured.calibration_eligible & np.isfinite(measured.arrival_execution_bp)].copy()
valid['date']=pd.to_datetime(valid.arrival_time,utc=True).dt.strftime('%Y-%m-%d')
if validation['status']=='evaluated':
    train=valid.loc[valid.date.le(validation['train_end'])]
    test=valid.loc[valid.date.ge(validation['test_start'])]
    train=train.loc[pd.to_datetime(train.end_time,utc=True).lt(pd.to_datetime(test.arrival_time,utc=True).min())]
    assert train.date.max()<test.date.min()
    assert not set(train.parent_order_id)&set(test.parent_order_id)
    prediction=test.side.map(train.groupby('side').arrival_execution_bp.median())
    assert np.isclose((test.arrival_execution_bp-prediction).abs().mean(),validation['baseline_test_mae_bp'])
    assert len(train)==validation['train_orders'] and len(test)==validation['test_orders']
for group in public['groups']:
    sample=measured.loc[measured.side.eq(group['side'])]
    for name,metric in group['metrics'].items():
        values=sample[name].dropna()
        assert len(values)==metric['count']>=public['definitions']['min_group_orders']
        assert np.isclose(values.median(),metric['p50'])
        assert np.isclose(values.quantile(.9),metric['p90'])
serialized=json.dumps(public)
assert 'parent_order_id' not in serialized and 'private-order' not in serialized
print({'holdout_status':validation['status'],'public_groups':len(public['groups'])})